In [ ]:
configfile = "config/config.yml"
input_data = "results/data/checkpoints/beforefilter_intermediate_empfaenger.pq"
targetpop_data = "results/data/checkpoints/targetpop.pq"
display_util = "workflow/scripts/display_util.py"
util = "/workflow/scripts/util.py"
output_data = "results/data/intermediate_empfaenger.pq"
output_model = "results/data/intermediate_empfaenger.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import rule_setup, display_data_doc, collist  # noqa: E402
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
    common_translate,
    split_data,
    find_redundant_cols,
    fix_redundancies,
    EmpfaengerID,
    collapse_col,
    replace_single_val_inplace,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = donors = collapse_col(
    data.loc[:, ["recipient_et_id_et", "recipient_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["recipient_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of recipients in the data ({rec.nunique()}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {rec[rec.isin(targetpop["recipient_et_id_et"])].nunique()} in the processed data.
        """
    )
)
del targetpop, rec

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`IQTIG` and {term}`ET` data is already connected (see [](general:ic)). The following table lists the different types of rows, which occur in this file and which ID combination they use, indicating the data contributors.

In [ ]:
idcols = ["recipient_et_id_et", "recipient_et_iqtig"]
df = split_data(data, idcols)
summar = split_data(data, idcols, return_summar=True)
assert len(df) == 2, "Not 2 different row types present!?"
del df, summar, idcols

After filtering to our target population there were no rows, which only contain data contributed from the {term}`ET`, most rows have data from both the {term}`ET` and the {term}`IQTIG`. The registry already joined the rows from the different institutes.

## Domain Steps

For this file the general plan for domain preprocessing was followed (see [](general:ds)).

### Row Filtering

No further filtering was necessary for this file. (see [](general:rf)).

### Unit Conversions

All measurements with units in this file use a single unit (see [](general:uc)). 

In [ ]:
cols = [
    "height_unit_et",
    "height_unit_iqtig",
    "weight_unit_et",
    "weight_unit_iqtig",
]
for col in cols:
    assert len(data[col].dropna().unique()) == 1
    data.drop(col, axis=1, inplace=True)
display(Markdown(f"""
Hence, we remove the unit columns {collist(cols)}.
"""))

Now we rename values for categorical variables so that they are equal among {term}`IQTIG` and {term}`ET` sourced columns.

In [ ]:
data = common_translate(data, config["data"]["common_translations"])
# Maybe add to config more if necessary

Some columns use 999 as an marker for invalid values. These values are replaced with the missing value indicator.

In [ ]:
height = "height_cm_et"
weight = "weight_kg_et"
replace_single_val_inplace(data, [height, weight])

In [ ]:
sel = data[height] < data[weight]
display(Markdown(f"""
Weights and heights appear to be sometimes mixed between `{height}` and `{weight}`. We assume that the height is higher than the weight. The height was smaller than the weight {sel.sum()} times and we switched the values for these recipients.
"""))

oldheights = data[height].copy()
data[height] = data[height].mask(sel, other=data[weight])
data[weight] = data[weight].mask(sel, other=oldheights)

### Consolidating Columns

We consolidated columns that appear for both {term}`ET` and {term}`IQTIG` (see [](general:crc)). As the rhesus column from {term}`IQTIG` is removed earlier, as it contains only missing values, only the ET column gets used and renamed to `rhesus`.

In [ ]:
red = find_redundant_cols(data)
red["recipient_et_id_et"] = [
    "recipient_et_id_et",
    "recipient_et_iqtig",
]
fix_redundancies(data, red)
# data.rename(columns={"recipient_id": "recipient_et_id_et"}, inplace=True)
assert len(data["recipient_et_id_et"].unique()) == len(
    data["recipient_et_id_et"]
), "Duplicate recipients!!"

data.rename(columns={"rhesus_et": "rhesus"}, inplace=True)

## Intermediate Dataset

In [ ]:
data = data.set_index("recipient_et_id_et").sort_index(axis=1).sort_index(axis=0)

In [ ]:
class Empfaenger(EmpfaengerID):
    birthdate: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Birthdate",
        description="When was the patient born?",
    )
    bloodgroup: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bloodgroup",
        description="What was the patients bloodgroup?",
        isin=["A", "0", "B", "AB"],
    )
    bloodtransfusion_after_reg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bloodtransfusion After Registration",
        description="Did the patient have a bloodtransfusion after the registration for a transplantation?",
        isin=["yes", "no"],
    )
    bloodtransfusion_before_reg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bloodtransfusion Before Registration",
        description="Did the patient have a bloodtransfusion before the registration for a transplantation?",
        isin=["yes", "no"],
    )
    children: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Children",
        description="How many children does the patient have?",
        ge=0,
    )
    death_cause: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Cause",
        description="Why did this patient die?",
    )
    death_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Death Date",
        description="When did this patient die?",
    )
    height_cm: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Height",
        description="How large was the patient?",
        ge=0,
    )
    nationality: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Nationality",
        description="What is the patients nationality?",
    )
    origin_country: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Origin country",
        description="What is the origin country of this patient?",
    )
    pregnancies: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pregnancies",
        description="How many times was the patient pregnant?",
        ge=0,
    )
    sex: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sex",
        description="What is the patients sex?",
        isin=["male", "female"],
    )
    rhesus: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Rhesus Factor",
        description="What is the patients rhesus factor?",
        isin=["positive", "negative"],
    )
    weight_kg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Weight",
        description="How much does this patient weigh in kg?",
        ge=0,
    )

    class Config:
        title = "Recipient Dataset"
        description = "Each row represents a (potential) recipient. The data is based on the 'element_empfaenger.csv' file. It contains data from ET and IQTIG."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(Empfaenger, data)

In [ ]:
Empfaenger.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    Empfaenger.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)